# Advanced Architectures and Applications in Agentic AI
## CA1 Assessment: Adaptive Customer Service Harness

Student name: Maximiliano Herrera </br>
Student ID: d00296830

## Dataset Preparation

### Load and Analysis

In [1]:
import pandas as pd

df = pd.read_csv('customer-dataset.csv')
df

,Unnamed: 0,flags,instruction,category,intent,response
0,0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...,...
26867,26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,26870,BL,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


In [2]:
categories = set(df['category'])
print(categories)
print(len(categories))

{'DELIVERY', 'SHIPPING', 'FEEDBACK', 'INVOICE', 'CONTACT', 'ORDER', 'CANCEL', 'ACCOUNT', 'REFUND', 'SUBSCRIPTION', 'PAYMENT'}
11


In [3]:
df.groupby(["category"]).size().reset_index(name="count")

,category,count
0,ACCOUNT,5986
1,CANCEL,950
2,CONTACT,1999
3,DELIVERY,1994
4,FEEDBACK,1997
5,INVOICE,1999
6,ORDER,3988
7,PAYMENT,1998
8,REFUND,2992
9,SHIPPING,1970


In [4]:
df.groupby(["category", "intent"]).size().reset_index(name="count")

,category,intent,count
0,ACCOUNT,create_account,997
1,ACCOUNT,delete_account,995
2,ACCOUNT,edit_account,1000
3,ACCOUNT,recover_password,995
4,ACCOUNT,registration_problems,999
5,ACCOUNT,switch_account,1000
6,CANCEL,check_cancellation_fee,950
7,CONTACT,contact_customer_service,1000
8,CONTACT,contact_human_agent,999
9,DELIVERY,delivery_options,995


### Filter dataset by categories and intents

In [5]:
categories_selected = ['ACCOUNT', 'ORDER', 'REFUND']
intents_selected = ['create_account', 'delete_account', 'edit_account',
                    'cancel_order', 'change_order', 'place_order',
                    'check_refund_policy', 'get_refund', 'track_refund'
                   ]
df_filtered = df[
    df['category'].isin(categories_selected) &
    df['intent'].isin(intents_selected)
].rename(columns={"Unnamed: 0": "id"})

print(f"df_filtered: {len(df_filtered)}, {type(df_filtered)}")

df_filtered: 8977, <class 'pandas.DataFrame'>


In [6]:
import plotly.express as px

def plot_number_records_per_intent(dataframe):
    fig = px.bar(
        dataframe['intent'].value_counts().reset_index(),
        x='intent',
        y='count',
        title='Number of Records per Intent',
        text_auto=True
    )

    fig.show()

plot_number_records_per_intent(df_filtered)

In [7]:
from pandas import DataFrame

def get_split_dataset(categories_selected, intents_selected, max_total_rows=60, random_state=42):
    intent_count = len(intents_selected)
    # Largest even count per intent that stays within the cap, so train and test match.
    rows_per_intent = (max_total_rows // intent_count) // 2 * 2
    rows_per_split = rows_per_intent // 2
    if rows_per_split < 1:
        raise ValueError("max_total_rows is too small for one train and one test row per intent")

    print(f"intents: {intent_count}")
    print(f"rows_per_intent: {rows_per_intent}")
    print(f"rows_per_split: {rows_per_split}")

    ds_filtered = df[
        df["category"].isin(categories_selected) &
        df["intent"].isin(intents_selected)
    ].rename(columns={"Unnamed: 0": "id"})
    print(f"ds_filtered: {len(ds_filtered)}")

    sampled = ds_filtered.groupby("intent", group_keys=False).sample(
        n=rows_per_intent, random_state=random_state
    )
    train_df = sampled.groupby("intent", group_keys=False).head(rows_per_split)
    test_df = sampled.drop(train_df.index)

    print(f"sampled: {len(sampled)}")
    print(f"train_df: {len(train_df)}")
    print(f"test_df: {len(test_df)}")
    return train_df, test_df

def get_dataset_batches_unbalanced(dataframe: DataFrame, batch_size=30, random_state=42):
    shuffled_df = (
        dataframe
        .drop_duplicates(subset=["instruction"]) # remove duplicated instructions
        .sample(frac=1, random_state=random_state) # shuffle records and use random_state for reproducibility
        .reset_index(drop=True) # reset index and do not insert it as new column
    )

    for start in range(0, len(shuffled_df), batch_size): #get a sequence from start (0, inclusive) to stop (len(shuffled_df), exclusive) with a step (batch_size)
        batch = shuffled_df.iloc[start:start + batch_size] # select batch_size rows starting from start

        #if batch is incomplete then break the loop
        if len(batch) < batch_size: 
            break

        # return a copy of the batch so original data is unchanged
        yield batch.copy()


def get_dataset_batches_balanced(dataframe: DataFrame, batch_size=30, random_state=42):    
    remaining = dataframe.drop_duplicates(subset=["instruction"]).copy()    
    num_intents = remaining["intent"].nunique()  
    # Largest even count per intent that stays within the cap, so train and test match.
    rows_per_intent =  batch_size// num_intents

    if rows_per_intent == 0:
        raise ValueError("batch_size must be at least the number of intents")

    num_batches = remaining.groupby("intent").size().min() # rows per intent
    for _ in range(num_batches):
        batch = remaining.groupby("intent").sample(
            n=rows_per_intent,
            random_state=random_state
        )

        remaining= remaining.drop(batch.index)

        yield batch

def plot_and_preview_dataset(batch: DataFrame):
    plot_number_records_per_intent(batch)
    print(f"Num rows: {len(batch)}, num intents: {batch["intent"].nunique()}")
    print(len(batch))        
         

In [8]:
batches = get_dataset_batches_balanced(df_filtered,30)
train1_df =next(batches)
train2_df =next(batches)
test1_df =next(batches)
test2_df =next(batches)

train1_df =train1_df.head(3)
train2_df =train2_df.head(3)
test1_df =test1_df.head(3)
test2_df =test2_df.head(3)

# train_df, test_df = get_split_dataset(categories_selected, intents_selected, 50)

In [9]:
plot_and_preview_dataset(train1_df)
train1_df

Num rows: 3, num intents: 1
3


,id,flags,instruction,category,intent,response
860,860,BILZ,can you help me camceling the purchase I made?,ORDER,cancel_order,I've been informed that you need assistance wi...
79,79,BIQZ,whag do i need to do to cancel order {{Order N...,ORDER,cancel_order,I understand your need for assistance in cance...
479,479,BC,"I bought some product, I want to cancel order ...",ORDER,cancel_order,I've come to understand that you've made a pur...


In [10]:
plot_and_preview_dataset(train2_df)
train2_df

Num rows: 3, num intents: 1
3


,id,flags,instruction,category,intent,response
863,863,BLQ,need assistance to cancel purchase {{Order Num...,ORDER,cancel_order,I acknowledge that you're seeking assistance w...
94,94,B,I want help canceling order {{Order Number}},ORDER,cancel_order,I appreciate your request for assistance in ca...
791,791,BCEILNQZ,i cannot pay for these product how do i cancel...,ORDER,cancel_order,I've realized you're unable to make payment fo...


In [11]:
plot_number_records_per_intent(test1_df)
test1_df

,id,flags,instruction,category,intent,response
78,78,BCL,I have got to cancel purchase {{Order Number}}...,ORDER,cancel_order,I've grasped that you need assistance with can...
321,321,BCILNPQ,i dont want purchase {{Order Number}} could i ...,ORDER,cancel_order,I grasp your desire to cancel your purchase wi...
801,801,BILZ,can you help me cancelling purchase {{Order N...,ORDER,cancel_order,I'll take care of it! I'm here to assist you w...


In [12]:
plot_number_records_per_intent(test2_df)
test2_df

,id,flags,instruction,category,intent,response
756,756,BLQZ,needassistance cancelling purchase {{Order Num...,ORDER,cancel_order,I've understood that you need assistance with ...
963,963,B,problem with canceling order {{Order Number}},ORDER,cancel_order,I realized that you're encountering difficulti...
221,221,BIL,what do I need to do to cancel purchase {{Orde...,ORDER,cancel_order,I gather that you're seeking guidance on cance...


## Test Harnes Agent

In [13]:
from general import clean_agentic_memory

clean_agentic_memory()

 * ./AGENT.md file created...


### Test inference mode, wihtout agentic memory

In [14]:
import logging
from general import initialize_logging
from harness_agent.harness import AgentHarness
from llm import get_llm

initialize_logging(level=logging.INFO)

# model = get_llm("qwen/qwen3.8-27b")
model = get_llm("openai/gpt-oss-120b")
harness = AgentHarness(model = model, training_dataset=train1_df, training_mode=False, agentic_memory_disabled=True)
# # response = harness.invoke("see the available payment options")
# # response = harness.invoke("were can i cancel purchase {{Order Number}}")
response = harness.invoke("can you help me camceling the purchase I made?")
response

2026-11-02 02:44:05 - harness.py - __post_init__ - INFO - Initializing AgentHarness - training_mode: False, agentic_memory_disabled: True
2026-11-02 02:44:05 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:05 - harness.py - invoke - INFO - Harness Response: Sure, I can help you get that purchase cancelled. To look up the order and start the cancellation process, I’ll need a few details:

1. **Order number** (or receipt/reference number)  
2. **Email address** you used when you placed the order  
3. The **date** of the purchase (if you have it)

Once I have that information, I can:

- Verify the order status (it needs to be still “pending” or “shipped” for a straightforward cancellation).  
- Initiate the cancellation on your behalf or give you the exact steps to do it in our portal.  
- Let you know about any refund timeline or restocking fees, if applicable.

Feel free to share the detai

{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='1d92b777-7cea-4671-a9b9-959856542d96'),
  AIMessage(content='Sure, I can help you get that purchase cancelled. To look up the order and start the cancellation process, I’ll need a few details:\n\n1. **Order number** (or receipt/reference number)  \n2. **Email address** you used when you placed the order  \n3. The **date** of the purchase (if you have it)\n\nOnce I have that information, I can:\n\n- Verify the order status (it needs to be still “pending” or “shipped” for a straightforward cancellation).  \n- Initiate the cancellation on your behalf or give you the exact steps to do it in our portal.  \n- Let you know about any refund timeline or restocking fees, if applicable.\n\nFeel free to share the details above (you can mask any part of the order number you’re uncomfortable posting publicly, e.g., “#1234‑****”). If you’d rather handle it privately, I 

### Test multiturn

In [15]:
response = harness.invoke("Unfortunatelly I cannot give you none of those details...")
response

2026-11-02 02:44:08 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:08 - harness.py - invoke - INFO - Harness Response: I completely understand — you don’t have to share any personal or order information here. You can still cancel the purchase on your own; here’s a step‑by‑step guide that works for most of our platforms:

---

## 1. Cancel Directly in the Online Store (Web or Mobile App)

| Step | What to Do | Why |
|------|------------|-----|
| **a. Sign in** | Log into the account you used when you placed the order. | The system needs to match the purchase to your profile. |
| **b. Go to “My Orders” / “Order History”** | Usually found under the account menu (icon → “My Account” → “Orders”). | This page lists every recent transaction. |
| **c. Find the order you want to cancel** | Look for the most recent order or use the date filter if you have many. | Identifies the exact purchase. |
|

{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='1d92b777-7cea-4671-a9b9-959856542d96'),
  AIMessage(content='Sure, I can help you get that purchase cancelled. To look up the order and start the cancellation process, I’ll need a few details:\n\n1. **Order number** (or receipt/reference number)  \n2. **Email address** you used when you placed the order  \n3. The **date** of the purchase (if you have it)\n\nOnce I have that information, I can:\n\n- Verify the order status (it needs to be still “pending” or “shipped” for a straightforward cancellation).  \n- Initiate the cancellation on your behalf or give you the exact steps to do it in our portal.  \n- Let you know about any refund timeline or restocking fees, if applicable.\n\nFeel free to share the details above (you can mask any part of the order number you’re uncomfortable posting publicly, e.g., “#1234‑****”). If you’d rather handle it privately, I 

In [16]:
response = harness.invoke("Can you remind me my first question please?")
response

2026-11-02 02:44:08 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:08 - harness.py - invoke - INFO - Harness Response: Your first question was:

**“can you help me camceling the purchase I made?”**


{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='1d92b777-7cea-4671-a9b9-959856542d96'),
  AIMessage(content='Sure, I can help you get that purchase cancelled. To look up the order and start the cancellation process, I’ll need a few details:\n\n1. **Order number** (or receipt/reference number)  \n2. **Email address** you used when you placed the order  \n3. The **date** of the purchase (if you have it)\n\nOnce I have that information, I can:\n\n- Verify the order status (it needs to be still “pending” or “shipped” for a straightforward cancellation).  \n- Initiate the cancellation on your behalf or give you the exact steps to do it in our portal.  \n- Let you know about any refund timeline or restocking fees, if applicable.\n\nFeel free to share the details above (you can mask any part of the order number you’re uncomfortable posting publicly, e.g., “#1234‑****”). If you’d rather handle it privately, I 

### Test training mode and enable agentic memory

In [17]:
from harness_agent.harness import AgentHarness
import logging
from llm import get_llm

initialize_logging(level=logging.INFO)

# model = get_llm("qwen/qwen3.8-27b")
model = get_llm("openai/gpt-oss-120b")
# model = get_llm("gpt-6-luna")
harness = AgentHarness(model = model, training_dataset=train1_df, training_mode=True, agentic_memory_disabled=False)
response = harness.invoke("can you help me camceling the purchase I made?")
response

2026-11-02 02:44:08 - harness.py - __post_init__ - INFO - Initializing AgentHarness - training_mode: True, agentic_memory_disabled: False
2026-11-02 02:44:09 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:09 - supervisor.py - request_supervisor - INFO - Supervisor called
2026-11-02 02:44:10 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:11 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:11 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:11 - skills_creator_agent.py - invoke - INFO - SkillsCreatorAgent executed successfully, number of tool calls: 1
2026-11-02 02:44:13 - _client.py - _send_single_

{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='73c74a79-4d90-4bcf-9e79-beb1c20c091d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants help canceling a purchase. No skills are listed. According to instructions, if no skill matches, must call request_supervisor with the user query, then produce answer in own words, not copying exactly. So we need to call request_supervisor.', 'tool_calls': [{'id': 'fc_5952ce64-36f6-490d-8da4-8fa17c40014a', 'function': {'arguments': '{"user_query":"can you help me camceling the purchase I made?"}', 'name': 'request_supervisor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 567, 'total_tokens': 660, 'completion_time': 0.193593896, 'completion_tokens_details': {'reasoning_tokens': 54}, 'prompt_time': 0.003307892, 'prompt_tokens_details': {'cached_tokens': 512}, 'queue_time': 0.004909744

### Test that multiturn should be disable on training mode

In [18]:
response = harness.invoke("What was my first question?")
response

2026-11-02 02:44:14 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:14 - supervisor.py - request_supervisor - INFO - Supervisor called
2026-11-02 02:44:15 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:16 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:17 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:17 - skills_creator_agent.py - invoke - INFO - SkillsCreatorAgent executed successfully, number of tool calls: 1
2026-11-02 02:44:18 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-02 02:44:19 - _client.py - _

{'messages': [HumanMessage(content='What was my first question?', additional_kwargs={}, response_metadata={}, id='60c07c55-c02a-4b42-9081-519c70aef4fb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What was my first question?" This is a meta question about prior conversation. There\'s no skill for that. No skill matches. So we must call request_supervisor with user_query.', 'tool_calls': [{'id': 'fc_29027539-b762-48ae-a3b8-50430af71660', 'function': {'arguments': '{"user_query":"What was my first question?"}', 'name': 'request_supervisor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 735, 'total_tokens': 810, 'completion_time': 0.156239023, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.460501203, 'prompt_tokens_details': {'cached_tokens': 256}, 'queue_time': 0.004820026, 'total_time': 0.616740226}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be

## Testing and Evaluation

In [19]:
clean_agentic_memory()

2026-11-02 02:44:19 - general.py - clean_agentic_memory - INFO - Cleaning agentic memory...
2026-11-02 02:44:19 - general.py - clean_agentic_memory - INFO -  * rmtree Skills...
2026-11-02 02:44:19 - general.py - clean_agentic_memory - INFO -  * .skills folder created...
2026-11-02 02:44:19 - general.py - clean_agentic_memory - INFO -  * Deleted AGENT.md...


 * ./AGENT.md file created...


In [20]:
from langsmith import Client
from langsmith.schemas import Dataset
from langsmith.utils import LangSmithConflictError

delete_if_exists = True

client = Client()

def create_dataset(name, description, delete_if_exists= False):
    try:
        if client.has_dataset(dataset_name=name):
            if delete_if_exists:
                print(f"Dataset '{name}' already exists. Deleting existing dataset...")
                client.delete_dataset(dataset_name=name)
            else:
                print(f"Dataset '{name}' already exists. Skipping and retrieving from Langsmith...")
                dataset = client.read_dataset(dataset_name=name)
                return dataset

        dataset = client.create_dataset(dataset_name = name, description = description)
        print(f"Dataset '{name}' created successfully.")
        
        return dataset       
    except LangSmithConflictError as e:
        print(f"Error on creating Dataset '{name}': {str(e)}")  

dataset_name = 'CA1 - Customer Dataset'
dataset = create_dataset(
    name = dataset_name,
    description='Customer questions and responses',
    delete_if_exists= delete_if_exists
)

Dataset 'CA1 - Customer Dataset' already exists. Deleting existing dataset...
Dataset 'CA1 - Customer Dataset' created successfully.


In [21]:
from pandas import DataFrame

def create_examples(dataset: Dataset, df: DataFrame, split_name: str = 'train'):
    print(f"Creating {len(df)} examples for split: {split_name}")

    example_ids = []
    for _,row in df.iterrows():
        example = client.create_example(
            dataset_id=dataset.id,            
            inputs= {
                "id": row["id"],
                "category": row["category"],
                "intent": row["intent"],
                "instruction": row["instruction"],
            },
            outputs= {
                "response": row["response"]                         
            }
        )

        example_ids.append(example.id)

    print(f"Set split name: {split_name}")
    client.update_examples(example_ids=example_ids,splits=[split_name] * len(example_ids))

In [22]:
SPLIT_TRAIN_1 = 'train_1'
SPLIT_TRAIN_2 = 'train_2'
SPLIT_TEST_1 = 'test_1'
SPLIT_TEST_2 = 'test_2'
create_examples(dataset, train1_df, SPLIT_TRAIN_1)
create_examples(dataset, train2_df, SPLIT_TRAIN_2)
create_examples(dataset, test1_df, SPLIT_TEST_1)
create_examples(dataset, test2_df, SPLIT_TEST_2)

Creating 3 examples for split: train_1
Set split name: train_1
Creating 3 examples for split: train_2
Set split name: train_2
Creating 3 examples for split: test_1
Set split name: test_1
Creating 3 examples for split: test_2
Set split name: test_2


In [23]:
import pandas as pd

def get_examples_by_split(dataset_name, split_name):    
    examples = list(
        client.list_examples(
            dataset_name=dataset_name,
            splits=[split_name]
        )
    )

    df_examples = pd.DataFrame([
        {
            **example.inputs, #unpack operator
            **example.outputs,
            "langsmith_id": str(example.id),
            "splits": example.metadata.get("dataset_split")
                    if example.metadata else None
        }
        for example in examples
    ])

    return {
        'name' : split_name,
        'examples' : examples,
        'df': df_examples
    }

# def get_examples_from_dataset(dataset_name):
#     train_split = get_examples_by_split(dataset_name, SPLIT_TRAIN_1N)
#     test_split = get_examples_by_split(dataset_name, SPLIT_TEST)
#     return {
#         'train_split': train_split,
#         'test_split': test_split,
#     }

# splits = get_examples_from_dataset(dataset_name)
# splits

split_train_1 = get_examples_by_split(dataset_name,SPLIT_TRAIN_1)
split_train_2 = get_examples_by_split(dataset_name,SPLIT_TRAIN_2)
split_test_1 = get_examples_by_split(dataset_name,SPLIT_TEST_1)
split_test_2 = get_examples_by_split(dataset_name,SPLIT_TEST_2)

In [24]:
from harness_agent.harness import AgentHarness
from pydantic import BaseModel, Field
from llm import get_llm
from langchain_core.messages import HumanMessage, SystemMessage

class ResponseEvaluatorSchema(BaseModel):
    """Class to define the schema for quality evaluation."""
    score: int = Field(description="An integer score from 1 to 10 indicating the quality of the generated answer.")
    comment: str = Field(description="A brief explanation of the score")

def evaluator_agent_response_quality(inputs: dict, outputs: dict, reference_outputs: dict, model_name:str) -> dict:
    """Evaluate the quality of the agent's generated response using LLM with a structured output."""  
    query = inputs["instruction"]
    agent_response = outputs['agent_response']
    reference_response = reference_outputs['response']

    system_eval_prompt = "You are a Expert Customer Service Agent Evaluator."
    eval_prompt =  f"""User query:
{query}

Reference answer:
{reference_response}

Agent response:
{agent_response}

Give the response a score from 1 to 10 and a short comment.
1 means the response is incorrect or unhelpful. 10 means it is fully correct and helpful.
Consider whether it:
- responds in the right way, such as asking a question or giving instructions;
- includes the important information from the reference;
- avoids unsupported information;
- is clear and uses an appropriate tone."""

    model = get_llm(model_name)
    result = model.with_structured_output(ResponseEvaluatorSchema).invoke([SystemMessage(system_eval_prompt), HumanMessage(eval_prompt)])
    
    evaluator_result = {"score": result.score, "comment": result.comment}
    logging.debug(f"evaluator_agent_response_quality: {evaluator_result}")

    return evaluator_result

def run_agent(query:str, model_name: str, training_dataset, training_mode: bool, agentic_memory_disabled : bool):      
    model = get_llm(model_name)
    harness = AgentHarness(model = model, training_dataset=training_dataset, training_mode=training_mode, agentic_memory_disabled=agentic_memory_disabled)
    # response = harness.invoke("see the available payment options")
    response = harness.invoke(query)
    agent_response = response['text']
    
    logging.debug("---------------------run_agent------------------------")
    logging.debug(f"response: {agent_response}")
    
    return agent_response

def target_function(inputs: str, model_name: str, training_dataset, training_mode: bool, agentic_memory_disabled : bool) -> dict:
    agent_response = run_agent(inputs["instruction"], model_name, training_dataset, training_mode, agentic_memory_disabled)
    logging.debug("---------------------target_function------------------------")
    logging.debug(f"agent_response: {agent_response}")
    
    return {"agent_response": agent_response}


In [25]:
def evaluate_agent(agent_model_name, judge_model_name,training, disable_agentic_memory, split):
    dataframe = split['df']
    examples = split['examples']
    training_mode = "training" if training else "testing"
    agentic_memory_mode  = "memory_disabled" if disable_agentic_memory else "memory_on"

    logging.debug(f"Evaluate Agent --> mode: {training_mode}, agentic_memory_mode: {agentic_memory_mode}")

    client.evaluate(
        lambda inputs: target_function(inputs, agent_model_name, dataframe, training, disable_agentic_memory),
        data= examples,
        evaluators=[lambda inputs, outputs, reference_outputs : evaluator_agent_response_quality( inputs, outputs, reference_outputs, judge_model_name )],
        experiment_prefix=f"exp-{training_mode}-{agentic_memory_mode}-{agent_model_name}",
        metadata={
            "model_name": agent_model_name,
            "training_mode": training_mode,
            "agentic_memory": agentic_memory_mode,
            "split": split["name"]
        },
        #Training must run sequentially because there are read/write operations into files
        max_concurrency= 1 if training else 5 
    )
    
# I will not traion the agent in this way (off-line from LangSmith) because the experiments will give me more obvservability on agent's behavior
def train_agent(model_name, split):
    dataframe = split['df']
    
    for row in dataframe.itertuples():
        agent_response = run_agent(query= row["instruction"], model_name = model_name, training_dataset= dataframe, training_mode =True  , agentic_memory_disabled=False)
    

In [26]:
#1 run in training mode with train split
#2 evaluate in test mode with test split (without agentic memory)
#3 evaluate in test mode with test split (with agentic memory)

# model_name = "openai/gpt-oss-120b"
# model_name = "deepseek"
# model_name = "gpt-6-luna"
# #1 evaluate in test mode with test split (without agentic memory)
# evaluate_agent(model_name, training=False, disable_agentic_memory=True, split=splits['test_split'])

from general import clean_agentic_memory

def evaluation_pipeline(agent_models, judge_model):
    for model in agent_models:
        logging.info(f"Executing evaluation pipeline for model: {model}...")   
        clean_agentic_memory()   

        #Baseline: Evaluate agent on inference mode with agentic memory disabled
        evaluate_agent(model, judge_model, training=False, disable_agentic_memory=True, split=split_test_1)
        
        #Training 1: Evaluate agent on training mode with agentic memory enabled        
        evaluate_agent(model, judge_model, training=True, disable_agentic_memory=False, split=split_train_1)
        
        #Test 1 (after training 1): Evaluate agent on inference mode with agentic memory enabled using split_test_1 
        evaluate_agent(model, judge_model, training=False, disable_agentic_memory=False, split=split_test_1)

        #Training 2: Evaluate agent on training mode with agentic memory enabled        
        evaluate_agent(model, judge_model, training=True, disable_agentic_memory=False, split=split_train_2)
        
        #Test 2 (after training 2): Evaluate agent on inference mode with agentic memory enabled using split_test_1 
        evaluate_agent(model, judge_model, training=False, disable_agentic_memory=False, split=split_test_1)
        
        #Test 3 (after training 2): Evaluate agent on inference mode with agentic memory enabled using split_test_2
        evaluate_agent(model, judge_model, training=False, disable_agentic_memory=False, split=split_test_2)

In [27]:
initialize_logging(logging.WARNING)

AGENT_MODELS = [
    "openai/gpt-oss-120b",
    "deepseek-flash",
    "gpt-6-luna",
]

JUDGE_MODEL = "gpt-5-nano" #"deepseek-v4-pro" failing

evaluation_pipeline(AGENT_MODELS, JUDGE_MODEL)

 * ./AGENT.md file created...
View the evaluation results for experiment: 'exp-testing-memory_disabled-openai/gpt-oss-120b-14be9be0' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/1c571a5b-1585-46ea-a8aa-233e17ee65cf/compare?selectedSessions=c7846b9c-ba20-48e3-ad7d-2d9db86037a4


View the evaluation results for experiment: 'exp-training-memory_on-openai/gpt-oss-120b-d9998f15' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/1c571a5b-1585-46ea-a8aa-233e17ee65cf/compare?selectedSessions=788ddb1d-f002-4eee-b444-ca049ca2c0fb


View the evaluation results for experiment: 'exp-testing-memory_on-openai/gpt-oss-120b-802ae83e' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/1c571a5b-1585-46ea-a8aa-233e17ee65cf/compare?selectedSessions=d323a806-c349-4cd6-ad56-6eaa6072272f


View the evaluation results for experiment: 'exp-training-memory_on-openai/gpt-oss-120b-0cb2b43b' at:
https://eu.smith.l

In [28]:
#2 run in training mode with train split
evaluate_agent(model_name, training_mode=True, disable_agentic_memory=False, split=splits['train_split'])

NameError: name 'model_name' is not defined

In [ ]:
#3 evaluate in test mode with test split (with agentic memory)
evaluate_agent(model_name, training_mode=False, disable_agentic_memory=False, split=splits['test_split'])

View the evaluation results for experiment: 'exp-f5c5ba6a' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/1b7e0dbd-abb8-4fa8-ad25-1ebd5026077a/compare?selectedSessions=b34f055f-809d-4e92-8736-ee1c858ef4e0


Initializing AgentHarness - training_mode: False, agentic_memory_disabled: False
middlewares: 2
before_agent 1 messages
before_agent 1 messages
------harness: Thank you for your interest in our payment options! We offer a variety of methods to suit your needs, including credit/debit cards, PayPal, bank transfers, Apple Pay, and Google Wallet.

You can view the full list of accepted payment methods on our website by going to the Payment page.

If you have any questions or need help with a particular option, just let us know – we’re here to make the process as smooth as possible!
---------------------run_agent------------------------
response: Thank you for your interest in our payment options! We offer a variety of methods to suit your needs, inclu